In [ ]:
### STAT 5P87 PROJECT ###

### WHAT DRIVES OXYGEN LEVEL IN THE GULF ###
### OF SAINT LAWRENCE  ###


In [ ]:
# Set the working Directory 

import os

os.chdir("/Users/srushtidesai/Desktop/School/COURSES/STAT 5P87/Group Project")
print(os.getcwd())

In [ ]:
# Load Scotia glider datasets

import pandas as pd

data0 = pd.read_csv("scotia_20180720_87_delayed_corrected_v4.csv")
data1 = pd.read_csv("scotia_20181113_95_delayed_corrected_v4.csv")
data2 = pd.read_csv("scotia_20190605_100_delayed_corrected_v4.csv")
data3 = pd.read_csv("scotia_20210719_136_delayed_corrected_v4.csv")
data4 = pd.read_csv("scotia_20220421_150_delayed_corrected_v4.csv")


In [ ]:
# Combine all datasets 
data = pd.concat([data0, data1, data2, data3, data4], ignore_index=True)

# Structure of data 
data.info()


In [ ]:
# Rename columns
data.columns = [
    "Time", "Latitude", "Longitude", "Depth",
    "Temperature", "Salinity", "Density",
    "Oxygen_concentration"
]


In [ ]:
# Pre-Processing the data 

# Filter non-negative oxygen values and valid depths
data = data[data["Oxygen_concentration"] >= 0]
data = data[data["Depth"] >= 0]

In [ ]:
# Convert Time column to datetime format
data["Time"] = pd.to_datetime(data["Time"], utc=True)

# Extract date and hour-level timestamp
data["Date"] = data["Time"].dt.date
data["Hour"] = data["Time"].dt.strftime("%Y-%m-%d %H")


In [ ]:
# Function to aggregate data by hour
def aggregate_by_hour(df):

    # Remove Time column because it cannot be averaged
    temp = df.drop(columns=["Time"])

    # Group by Hour and Date and compute mean
    # for all numeric variables
    aggregated = (
        temp.groupby(["Hour", "Date"])
        .mean(numeric_only=True)
        .reset_index()
    )

    # Convert Hour back to datetime format
    aggregated["Hour"] = pd.to_datetime(aggregated["Hour"], utc=True)

    return aggregated


In [ ]:
# Apply hourly aggregation
aggregated_data = aggregate_by_hour(data)

# Extract month from Hour variable
aggregated_data["Month"] = aggregated_data["Hour"].dt.month


In [ ]:
# Assign seasons based on month
def get_season(m):
    if m in [4, 5, 6, 7]:
        return "Summer"
    elif m in [8, 9, 10]:
        return "Fall"
    else:
        return "Winter"

aggregated_data["Season"] = aggregated_data["Month"].apply(get_season)

In [ ]:
# Select relevant columns
data = aggregated_data.drop(columns=["Hour", "Date", "Latitude", "Longitude"])


# Classify into depths
data["Oxygen_concentration"] = data["Oxygen_concentration"].apply(
    lambda x: "High" if x >= 250 else "Low"
)

In [ ]:
# Save processed dataset
data.to_csv("scotia_data_hourly.csv", index=False)